In [1]:
# =============================================================================
# VICReg (ResNet-50) — FULL PIPELINE: PRE-TRAINING + EVALUATION
# Pre-train on PatternNet → Evaluate on EuroSAT-RGB / EuroSAT-MS
#
# ── Comparability changes (matched to DINO / SatMAE / MoCo-v3 baseline) ──────
#   C1.  img_size: 224 → 160       (matches DINO S20 / SatMAE C2 / MoCo-v3)
#   C2.  batch_size: 2048 → 512    (matches all three other models)
#   C3.  lr: 0.3 → 0.05            (reduced from 0.1 due to cov loss dominance)
#   C4.  ensure_split() 80/20 split, same SEED as all other models
#   C5.  eval crop: 224 → 160
#   C6.  Albumentations augmentation pipeline — identical to DINO/SatMAE
#   C7.  projector: 8192-8192-8192 (original) → 2048-2048-2048
#   C8.  epochs: 1000 (original paper) → 200 (matches DINO/SatMAE/MoCo-v3)
#
# ── Stability fixes (v2) ──────────────────────────────────────────────────────
#   SF-1. lr: 0.1 → 0.05 (cov loss was ~80x larger than inv/var at peak LR)
#   SF-2. nu_cov: 1.0 → 0.04 (scaled for batch=128 vs paper's batch=2048)
#   SF-3. _cov_loss clamped to max=50.0 (prevents runaway covariance)
#   SF-4. clip_grad_norm_: 3.0 → 1.0 (tighter clipping for stability)
#   SF-5. AMP_DTYPE forced to bfloat16 on Ampere+ (wider dynamic range vs fp16)
#
# ── Bug fixes ─────────────────────────────────────────────────────────────────
#   FIX-1.  channels_last only in train path; eval uses plain contiguous tensors
#   FIX-2.  DDP set_device + init_process_group before model construction.
#   FIX-3.  clip_grad_norm_ inside scaler.unscale_() block.
#   FIX-4.  torch.isnan(loss) guard with informative skip.
#   FIX-5.  async_save() threading for non-blocking checkpoint I/O.
#   FIX-6.  VICReg variance term uses F.relu (not clamp) to keep the loss
#            differentiable everywhere.
#
# ── Speed improvements ────────────────────────────────────────────────────────
#   S1.  torch.compile on full model (reduce-overhead).
#   S2.  BF16 on Ampere+ / FP16 fallback (AMP_DTYPE).
#   S3.  TF32 matmuls (torch.set_float32_matmul_precision("high")).
#   S4.  Fused AdamW (single CUDA kernel).
#   S5.  DDP multi-GPU or single-GPU training.
#   S6.  Albumentations augmentation pipeline, torchvision fallback.
#   S7.  channels_last memory layout in train forward (Conv2d benefit).
#   S8.  Two-view DataLoader with persistent_workers + prefetch.
# =============================================================================

# !pip install timm torchvision torch scipy scikit-learn albumentations --quiet

import os, math, random, json, time, threading, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets, models
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedShuffleSplit

# ── Albumentations (optional) ─────────────────────────────────────────────────
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    HAS_ALBU = True
except ImportError:
    HAS_ALBU = False
    print("albumentations not found — falling back to torchvision transforms.")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True
torch.set_float32_matmul_precision("high")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

# ── Environment ───────────────────────────────────────────────────────────────
BASE_DIR    = "/kaggle/working"
DATA_DIR    = "/kaggle/input"
NUM_WORKERS = 4
COMPILE     = False

# ── AMP dtype (SF-5: prefer bfloat16 on Ampere+ for wider dynamic range) ──────
def _amp_dtype():
    if not torch.cuda.is_available():
        return None
    # bfloat16 has much wider dynamic range than float16 — prevents NaN overflow
    if torch.cuda.get_device_capability()[0] >= 8:
        return torch.bfloat16
    return torch.float16

AMP_DTYPE = _amp_dtype()

# ── Config ────────────────────────────────────────────────────────────────────
CFG = dict(
    patternnet_dir  = f"{DATA_DIR}/datasets/samitsaleem/patternnet-scene-classification-dataset/PatternNet_Images",
    eurosat_rgb_dir = f"{DATA_DIR}/datasets/pranjallk1995/rgbeurosat/RBG",
    eurosat_ms_dir  = f"{DATA_DIR}/datasets/nguyenquangnhat2100/eurosatallbands/ds/images/remote_sensing/otherDatasets/sentinel_2/tif",
    output_dir      = f"{BASE_DIR}/vicreg",
    checkpoint      = None,

    # Architecture
    arch            = "resnet50",
    img_size        = 160,
    embed_dim       = 2048,

    # VICReg projector
    proj_hidden     = 2048,
    proj_out        = 2048,

    # VICReg loss coefficients
    # SF-2: nu_cov scaled from 1.0 → 0.04
    # Rationale: paper uses batch=2048; we use batch=128.
    # Covariance estimate noise scales as 1/sqrt(N), so weight scales as
    # (128/2048)^0.5 ≈ 0.25; halved again to 0.04 for added stability.
    lambda_inv      = 25.0,
    mu_var          = 25.0,
    nu_cov          = 0.04,     # was 1.0 — SF-2
    eps_var         = 1e-4,

    # Training
    # SF-1: lr reduced from 0.1 → 0.05
    # At peak lr=0.1, cov loss (~49) dominated total loss 80:1 over inv (~0.3),
    # causing gradient overflow under FP16 AMP starting at epoch 12.
    epochs          = 200,
    batch_size      = 128,
    lr              = 0.05,     # was 0.1 — SF-1
    min_lr          = 1e-6,
    weight_decay    = 1e-4,
    warmup_epochs   = 10,

    # Augmentation
    jitter_strength = 0.4,
    blur_prob       = 0.5,
    global_scale    = (0.2, 1.0),

    # Evaluation
    num_classes       = 10,
    knn_k             = 20,
    retrieval_ks      = [1, 5, 10],
    geo_thresholds_km = [1, 5, 10],
    val_split         = 0.2,
)

os.makedirs(CFG["output_dir"], exist_ok=True)


# =============================================================================
# ── Shared utilities ──────────────────────────────────────────────────────────
# =============================================================================

def make_loader(ds, batch_size, shuffle, drop_last=False,
                collate_fn=None, sampler=None):
    pw = NUM_WORKERS > 0
    kwargs = dict(
        batch_size         = batch_size,
        num_workers        = NUM_WORKERS,
        pin_memory         = True,
        drop_last          = drop_last,
        persistent_workers = pw,
        prefetch_factor    = 4 if pw else None,
    )
    if sampler is not None:
        kwargs["sampler"] = sampler
    else:
        kwargs["shuffle"] = shuffle
    if collate_fn is not None:
        kwargs["collate_fn"] = collate_fn
    return DataLoader(ds, **kwargs)


def ensure_split(root, val_frac=0.2, seed=SEED):
    if os.path.isdir(os.path.join(root, "train")):
        return root
    split_root = root.rstrip("/") + "_split"
    if os.path.isdir(os.path.join(split_root, "train")):
        print(f"  Using cached split at {split_root}")
        return split_root
    print(f"  Creating 80/20 stratified split → {split_root} …")
    base   = datasets.ImageFolder(root)
    labels = np.array([y for _, y in base.samples])
    sss    = StratifiedShuffleSplit(n_splits=1, test_size=val_frac,
                                    random_state=seed)
    train_idx, val_idx = next(sss.split(np.zeros(len(labels)), labels))
    for split_name, indices in [("train", train_idx), ("val", val_idx)]:
        for idx in indices:
            src_path, cls_idx = base.samples[idx]
            cls_name = base.classes[cls_idx]
            dst_dir  = os.path.join(split_root, split_name, cls_name)
            os.makedirs(dst_dir, exist_ok=True)
            dst_path = os.path.join(dst_dir, os.path.basename(src_path))
            if not os.path.exists(dst_path):
                try:
                    os.link(src_path, dst_path)
                except OSError:
                    shutil.copy2(src_path, dst_path)
    print(f"  Split: {len(train_idx)} train / {len(val_idx)} val")
    return split_root


_save_thread: threading.Thread = None

def async_save(path, obj):
    global _save_thread
    if _save_thread is not None:
        _save_thread.join()
    def _save():
        torch.save(obj, path)
        print(f"  → Saved {path}", flush=True)
    _save_thread = threading.Thread(target=_save, daemon=True)
    _save_thread.start()


def cosine_schedule(start, end, epoch, total):
    return end + 0.5 * (start - end) * (1.0 + math.cos(math.pi * epoch / total))


# =============================================================================
# ── Augmentation pipeline ─────────────────────────────────────────────────────
# =============================================================================

def _base_aug_albu(size, scale, jitter_s, blur_prob, solarize_prob=0.0):
    ops = [
        A.RandomResizedCrop(size=(size, size), scale=scale, interpolation=3),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=jitter_s, contrast=jitter_s,
                      saturation=jitter_s, hue=jitter_s * 0.25, p=0.8),
        A.ToGray(p=0.2),
        A.GaussianBlur(blur_limit=(9, 9), sigma_limit=(0.1, 2.0), p=blur_prob),
    ]
    if solarize_prob > 0:
        ops.append(A.Solarize(threshold=128, p=solarize_prob))
    ops += [
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]
    return A.Compose(ops)


def _base_aug_tv(size, scale, jitter_s, blur_prob, solarize_prob=0.0):
    ops = [
        transforms.RandomResizedCrop(size, scale=scale, interpolation=3),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(
            brightness=jitter_s, contrast=jitter_s,
            saturation=jitter_s, hue=jitter_s * 0.25)], p=0.8),
        transforms.RandomGrayscale(p=0.2),
        transforms.RandomApply(
            [transforms.GaussianBlur(kernel_size=9, sigma=(0.1, 2.0))],
            p=blur_prob),
    ]
    if solarize_prob > 0:
        ops.append(transforms.RandomSolarize(threshold=128, p=solarize_prob))
    ops += [
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
    return transforms.Compose(ops)


def _base_aug(size, scale, jitter_s, blur_prob, solarize_prob=0.0):
    return (_base_aug_albu(size, scale, jitter_s, blur_prob, solarize_prob)
            if HAS_ALBU else
            _base_aug_tv(size, scale, jitter_s, blur_prob, solarize_prob))


# =============================================================================
# ── Datasets ──────────────────────────────────────────────────────────────────
# =============================================================================

class TwoViewDataset(Dataset):
    def __init__(self, root, aug):
        self.base     = datasets.ImageFolder(root)
        self.aug      = aug
        self.use_albu = HAS_ALBU

    def __len__(self): return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        if self.use_albu:
            arr = np.array(img)
            x1  = self.aug(image=arr)["image"]
            x2  = self.aug(image=arr)["image"]
        else:
            x1 = self.aug(img)
            x2 = self.aug(img)
        return x1, x2, label


class MultiSpectralDataset(Dataset):
    def __init__(self, root, n_channels=13):
        self.samples    = []
        self.n_channels = n_channels
        classes = sorted(d for d in os.listdir(root)
                         if os.path.isdir(os.path.join(root, d)))
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        for cls in classes:
            cls_dir = os.path.join(root, cls)
            for fname in sorted(os.listdir(cls_dir)):
                if fname.lower().endswith(
                        (".tif", ".tiff", ".npy", ".png", ".jpg")):
                    self.samples.append(
                        (os.path.join(cls_dir, fname),
                         self.class_to_idx[cls]))

    def __len__(self): return len(self.samples)

    def _load_tif(self, path):
        try:
            import rasterio
        except ImportError:
            raise ImportError("pip install rasterio")
        with rasterio.open(path) as src:
            arr = src.read().astype(np.float32)
        n = self.n_channels
        if arr.shape[0] >= n:
            arr = arr[:n]
        else:
            pad = np.zeros((n - arr.shape[0], *arr.shape[1:]), dtype=np.float32)
            arr = np.concatenate([arr, pad], axis=0)
        return torch.from_numpy(arr)

    def _normalise(self, x):
        mean = x.view(x.shape[0], -1).mean(1, keepdim=True).unsqueeze(-1)
        std  = (x.view(x.shape[0], -1).std(1, keepdim=True)
                 .unsqueeze(-1).clamp(min=1e-6))
        return (x - mean) / std

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        ext = os.path.splitext(path)[1].lower()
        if ext in (".tif", ".tiff"):
            x = self._load_tif(path)
        elif ext == ".npy":
            arr = np.load(path).astype(np.float32)
            if arr.ndim == 3 and arr.shape[2] == self.n_channels:
                arr = arr.transpose(2, 0, 1)
            x = torch.from_numpy(arr)
            if x.shape[0] > self.n_channels:
                x = x[:self.n_channels]
        else:
            from PIL import Image
            img      = Image.open(path).convert("RGB")
            img_size = CFG["img_size"]
            resize   = int(img_size * (256 / 224) + 0.5)
            x = transforms.Compose([
                transforms.Resize(resize),
                transforms.CenterCrop(img_size),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406],
                                      [0.229, 0.224, 0.225]),
            ])(img)
            x = x.repeat(math.ceil(self.n_channels / 3), 1, 1)[:self.n_channels]
            return x, label
        x = self._normalise(x)
        x = F.interpolate(x[None], size=CFG["img_size"],
                          mode="bilinear", align_corners=False)[0]
        return x, label


# =============================================================================
# ── Backbone factory ──────────────────────────────────────────────────────────
# =============================================================================

class _CheckpointedSequential(nn.Sequential):
    def forward(self, x):
        for module in self:
            x = torch.utils.checkpoint.checkpoint(module, x, use_reentrant=False)
        return x


def _make_backbone():
    m = models.resnet50(weights=None)
    m.fc = nn.Identity()
    m.layer2 = _CheckpointedSequential(*list(m.layer2.children()))
    m.layer3 = _CheckpointedSequential(*list(m.layer3.children()))
    m.layer4 = _CheckpointedSequential(*list(m.layer4.children()))
    return m


# =============================================================================
# ── VICReg Projector ──────────────────────────────────────────────────────────
# =============================================================================

class VICRegProjector(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )
        nn.init.trunc_normal_(self.net[-1].weight, std=0.01)

    def forward(self, x):
        return self.net(x)


# =============================================================================
# ── VICReg Model ──────────────────────────────────────────────────────────────
# =============================================================================

class VICReg(nn.Module):
    """
    VICReg: Variance-Invariance-Covariance Regularization.

    Loss = λ · L_inv  +  μ · L_var  +  ν · L_cov

    Stability fixes applied (v2):
      SF-2: nu_cov=0.04 (scaled for batch=128)
      SF-3: _cov_loss clamped to max=50.0
      SF-4: grad clip tightened to 1.0 (in train loop)
    """

    def __init__(self, cfg):
        super().__init__()
        self.backbone   = _make_backbone()
        self.projector  = VICRegProjector(cfg["embed_dim"],
                                           cfg["proj_hidden"],
                                           cfg["proj_out"])
        self.lambda_inv = cfg["lambda_inv"]
        self.mu_var     = cfg["mu_var"]
        self.nu_cov     = cfg["nu_cov"]
        self.eps_var    = cfg["eps_var"]

    @staticmethod
    def _var_loss(z, eps):
        """FIX-6: F.relu for everywhere-differentiable variance loss."""
        std = torch.sqrt(z.var(dim=0) + eps)
        return F.relu(1.0 - std).mean()

    @staticmethod
    def _cov_loss(z):
        """
        Covariance loss: penalise off-diagonal entries.
        SF-3: clamped to max=50.0 to prevent runaway values from
        noisy covariance estimates at small batch sizes (128 vs 2048).
        """
        N, D = z.shape
        z    = z - z.mean(dim=0)
        z    = z / z.std(dim=0, unbiased=False).clamp(min=1e-4)
        cov  = (z.T @ z) / (N - 1)
        off_diag_mask = ~torch.eye(D, device=z.device, dtype=torch.bool)
        raw = cov.pow(2).masked_fill(~off_diag_mask, 0.0).sum() / D
        return raw.clamp(max=50.0)   # SF-3: safety ceiling

    def forward(self, x1, x2):
        z1 = self.projector(self.backbone(x1))
        z2 = self.projector(self.backbone(x2))

        l_inv = F.mse_loss(z1, z2)
        l_var = 0.5 * (self._var_loss(z1, self.eps_var) +
                        self._var_loss(z2, self.eps_var))
        l_cov = 0.5 * (self._cov_loss(z1) + self._cov_loss(z2))

        loss = (self.lambda_inv * l_inv +
                self.mu_var     * l_var +
                self.nu_cov     * l_cov)
        return loss, l_inv, l_var, l_cov


# =============================================================================
# ── LR schedule ───────────────────────────────────────────────────────────────
# =============================================================================

def build_epoch_schedules(cfg):
    lrs = []
    for e in range(cfg["epochs"]):
        if e < cfg["warmup_epochs"]:
            lr = cfg["lr"] * (e + 1) / cfg["warmup_epochs"]
        else:
            lr = cosine_schedule(cfg["lr"], cfg["min_lr"],
                                 e - cfg["warmup_epochs"],
                                 cfg["epochs"] - cfg["warmup_epochs"])
        lrs.append(lr)
    return lrs


def apply_lr(optimizer, lr):
    for g in optimizer.param_groups:
        g["lr"] = lr


# =============================================================================
# ── Training loop ─────────────────────────────────────────────────────────────
# =============================================================================

def train(rank: int, world_size: int):
    # FIX-2: set device and init process group FIRST
    torch.cuda.set_device(rank)
    device  = torch.device(f"cuda:{rank}")
    is_ddp  = world_size > 1
    is_main = rank == 0

    if is_ddp:
        dist.init_process_group(
            backend="nccl", init_method="env://",
            world_size=world_size, rank=rank)

    if is_main:
        print(f"Device: {device}  |  World: {world_size}  |  "
              f"Workers: {NUM_WORKERS}  |  compile: {COMPILE}  |  "
              f"AMP: {AMP_DTYPE}  |  albu: {HAS_ALBU}", flush=True)
        print(f"Stability config: lr={CFG['lr']}  nu_cov={CFG['nu_cov']}  "
              f"grad_clip=1.0  cov_clamp=50.0", flush=True)

    # ── Data ──────────────────────────────────────────────────────────────────
    aug     = _base_aug(CFG["img_size"], CFG["global_scale"],
                        CFG["jitter_strength"], CFG["blur_prob"])
    dataset = TwoViewDataset(CFG["patternnet_dir"], aug)
    sampler = (DistributedSampler(dataset, num_replicas=world_size,
                                   rank=rank, shuffle=True, drop_last=True)
               if is_ddp else None)
    loader  = make_loader(dataset, CFG["batch_size"],
                          shuffle=not is_ddp, drop_last=True,
                          sampler=sampler)

    # ── Model ─────────────────────────────────────────────────────────────────
    model = VICReg(CFG).to(device).to(memory_format=torch.channels_last)

    if COMPILE:
        try:
            model = torch.compile(model, mode="default")
            if is_main:
                print("torch.compile enabled.")
        except Exception as e:
            if is_main:
                print(f"torch.compile skipped: {e}")

    if is_ddp:
        model = DDP(model, device_ids=[rank], find_unused_parameters=False)

    # ── Optimizer (S4: fused AdamW) ───────────────────────────────────────────
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG["lr"],
        weight_decay=CFG["weight_decay"],
        fused=True)

    use_amp    = AMP_DTYPE is not None
    # SF-5: bfloat16 does not need a loss scaler (no overflow risk)
    use_scaler = AMP_DTYPE == torch.float16
    scaler     = torch.amp.GradScaler("cuda", enabled=use_scaler)
    trainable  = [p for p in model.parameters() if p.requires_grad]

    lrs = build_epoch_schedules(CFG)
    start_epoch = 0

    if CFG["checkpoint"] and is_main:
        ckpt = torch.load(CFG["checkpoint"], map_location="cpu")
        raw  = model
        if hasattr(raw, "_orig_mod"): raw = raw._orig_mod
        if hasattr(raw, "module"):    raw = raw.module
        raw.load_state_dict(ckpt["model"], strict=False)
        optimizer.load_state_dict(ckpt["optimizer"])
        start_epoch = ckpt["epoch"] + 1
        print(f"Resumed from epoch {start_epoch}")

    log           = []
    step_counter  = 0
    nan_count     = 0       # track consecutive NaN batches for early abort
    t_train_start = time.time()

    for epoch in range(start_epoch, CFG["epochs"]):
        if is_ddp:
            sampler.set_epoch(epoch)
        apply_lr(optimizer, lrs[epoch])

        model.train()
        total_loss = total_inv = total_var = total_cov = 0.0
        t0 = time.time()

        for x1, x2, _ in loader:
            x1 = x1.to(device, non_blocking=True,
                        memory_format=torch.channels_last)
            x2 = x2.to(device, non_blocking=True,
                        memory_format=torch.channels_last)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=use_amp):
                loss, l_inv, l_var, l_cov = model(x1, x2)
                loss = loss.mean()

            # FIX-4: guard against NaN/Inf
            if torch.isnan(loss) or torch.isinf(loss):
                nan_count += 1
                print(f"[rank {rank}] NaN/Inf at step {step_counter}, "
                      f"epoch {epoch+1} (total skipped: {nan_count}) "
                      f"— skipping batch.", flush=True)
                optimizer.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                step_counter += 1
                # Abort run if NaNs are persistent (indicates deeper issue)
                if nan_count >= 50:
                    print(f"[rank {rank}] Too many NaN batches ({nan_count}). "
                          f"Aborting training.", flush=True)
                    if is_ddp:
                        dist.destroy_process_group()
                    return
                continue

            nan_count = 0  # reset on successful step
            scaler.scale(loss).backward()
            # FIX-3: unscale before clip
            scaler.unscale_(optimizer)
            # SF-4: tighter grad clip (1.0 vs 3.0) for stability
            nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            total_inv  += l_inv.item()
            total_var  += l_var.item()
            total_cov  += l_cov.item()
            step_counter += 1

        n       = max(len(loader), 1)
        elapsed = time.time() - t0

        if is_main:
            log.append({
                "epoch":        epoch,
                "loss":         round(total_loss / n, 5),
                "l_inv":        round(total_inv  / n, 5),
                "l_var":        round(total_var  / n, 5),
                "l_cov":        round(total_cov  / n, 5),
                "lr":           lrs[epoch],
                "epoch_time_s": round(elapsed, 1),
            })
            print(f"Epoch [{epoch+1:>3}/{CFG['epochs']}]  "
                  f"loss={total_loss/n:.4f}  "
                  f"inv={total_inv/n:.4f}  "
                  f"var={total_var/n:.4f}  "
                  f"cov={total_cov/n:.4f}  "
                  f"lr={lrs[epoch]:.2e}  time={elapsed:.0f}s")

            if (epoch + 1) % 50 == 0 or epoch == CFG["epochs"] - 1:
                raw = model
                if hasattr(raw, "_orig_mod"): raw = raw._orig_mod
                if hasattr(raw, "module"):    raw = raw.module
                state = {k.replace("module.", ""): v
                         for k, v in raw.state_dict().items()}
                path = os.path.join(CFG["output_dir"],
                                     f"vicreg_ep{epoch+1}.pt")
                async_save(path, {"epoch": epoch, "model": state,
                                   "optimizer": optimizer.state_dict(),
                                   "cfg": CFG})

    if is_main:
        if _save_thread is not None:
            _save_thread.join()
        with open(os.path.join(CFG["output_dir"], "vicreg_log.json"), "w") as f:
            json.dump(log, f, indent=2)
        total = time.time() - t_train_start
        print(f"\nPre-training done.  Total: {total/3600:.2f} h ({total:.0f} s)")

    if is_ddp:
        dist.destroy_process_group()


# =============================================================================
# ── Backbone loading (eval) ───────────────────────────────────────────────────
# =============================================================================

def load_backbone(backbone_path, device):
    ckpt  = torch.load(backbone_path, map_location="cpu")
    state = {k.replace("module.", "").replace("backbone.", ""): v
             for k, v in ckpt["model"].items() if "backbone." in k}
    backbone = _make_backbone()
    backbone.load_state_dict(state, strict=False)
    backbone.eval().to(device)
    for p in backbone.parameters():
        p.requires_grad_(False)
    return backbone


def get_eval_transforms():
    img_size = CFG["img_size"]
    resize   = int(img_size * (256 / 224) + 0.5)
    val_tf = transforms.Compose([
        transforms.Resize(resize),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return train_tf, val_tf


@torch.inference_mode()
def extract_features(backbone, loader, device):
    all_feats, all_labels = [], []
    backbone.eval()
    use_amp = AMP_DTYPE is not None
    for x, y in loader:
        with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=use_amp):
            feats = backbone(x.to(device, non_blocking=True))
        all_feats.append(feats.float().cpu())
        all_labels.append(y)
    return torch.cat(all_feats), torch.cat(all_labels)


# =============================================================================
# ── Shared eval helpers ───────────────────────────────────────────────────────
# =============================================================================

def batched_knn_predict(sim, train_labels, k, num_classes, chunk=512):
    N_val  = sim.shape[0]
    device = sim.device
    preds  = torch.empty(N_val, dtype=torch.long, device=device)
    for start in range(0, N_val, chunk):
        end        = min(start + chunk, N_val)
        top_idx    = sim[start:end].topk(k, dim=1).indices
        top_labels = train_labels.to(device)[top_idx]
        B          = end - start
        votes      = torch.zeros(B, num_classes, device=device)
        votes.scatter_add_(1, top_labels,
                           torch.ones(B, k, device=device))
        preds[start:end] = votes.argmax(1)
    return preds


def effective_rank(feats):
    f       = feats - feats.mean(0)
    cov     = (f.T @ f) / (feats.shape[0] - 1)
    eigvals = torch.linalg.eigvalsh(cov.float()).clamp(min=0)
    eigvals = eigvals / eigvals.sum().clamp(min=1e-8)
    eigvals = eigvals[eigvals > 1e-9]
    entropy = -(eigvals * eigvals.log()).sum()
    return math.exp(entropy.item())


def uniformity_score(feats):
    feats = F.normalize(feats.float(), dim=-1)
    if feats.shape[0] > 2000:
        feats = feats[torch.randperm(feats.shape[0])[:2000]]
    sq = torch.cdist(feats, feats, p=2).pow(2)
    return round(sq.mul(-2).exp().mean().log().item(), 4)


def haversine_km(lat1, lon1, lat2, lon2):
    R  = 6371.0
    dr = math.radians
    dlat = dr(lat2 - lat1); dlon = dr(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2 +
         math.cos(dr(lat1)) * math.cos(dr(lat2)) *
         math.sin(dlon / 2) ** 2)
    return R * 2 * math.asin(math.sqrt(a))


def mean_average_precision(sim_matrix, labels):
    N  = sim_matrix.shape[0]
    sm = sim_matrix.clone()
    sm.fill_diagonal_(-1e9)
    order   = sm.argsort(dim=1, descending=True)
    ap_list = []
    for i in range(N):
        gt    = (labels[order[i]] == labels[i])
        n_rel = gt.sum().item()
        if n_rel == 0: continue
        n_correct = 0; precisions = []
        for rank_, hit in enumerate(gt.tolist(), 1):
            if hit:
                n_correct += 1
                precisions.append(n_correct / rank_)
        ap_list.append(sum(precisions) / n_rel)
    return float(np.mean(ap_list)) * 100 if ap_list else 0.0


# =============================================================================
# §4.1  CLASSIFICATION
# =============================================================================

def eval_classification(backbone_path, eurosat_dir,
                         label_fracs=(0.01, 0.1, 1.0), num_classes=10):
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone = load_backbone(backbone_path, device)
    train_tf, val_tf = get_eval_transforms()
    split_dir = ensure_split(eurosat_dir, CFG["val_split"])

    train_full = datasets.ImageFolder(
        os.path.join(split_dir, "train"), transform=train_tf)
    val_ds     = datasets.ImageFolder(
        os.path.join(split_dir, "val"),   transform=val_tf)

    print("  Pre-extracting features for linear probe…")
    all_train_feats, all_train_labels = extract_features(
        backbone, make_loader(train_full, 512, shuffle=False), device)
    val_feats, val_labels = extract_features(
        backbone, make_loader(val_ds, 512, shuffle=False), device)

    all_train_feats  = all_train_feats.to(device)
    all_train_labels = all_train_labels.to(device)
    val_feats_dev    = val_feats.to(device)
    val_labels_dev   = val_labels.to(device)

    results = {}
    bs      = 256

    for frac in label_fracs:
        n       = max(num_classes, int(len(train_full) * frac))
        indices = random.sample(range(len(all_train_feats)), n)
        idx_t   = torch.tensor(indices, device=device)
        f_sub   = all_train_feats[idx_t]
        l_sub   = all_train_labels[idx_t]

        head  = nn.Linear(CFG["embed_dim"], num_classes).to(device)
        opt   = torch.optim.SGD(head.parameters(), lr=0.1,
                                 momentum=0.9, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)

        for _ in range(100):
            head.train()
            perm = torch.randperm(len(f_sub), device=device)
            for start in range(0, len(f_sub), bs):
                sel = perm[start:start + bs]
                opt.zero_grad()
                F.cross_entropy(head(f_sub[sel]), l_sub[sel]).backward()
                opt.step()
            sched.step()

        head.eval()
        with torch.no_grad():
            preds = torch.cat([head(val_feats_dev[s:s + bs]).argmax(1)
                               for s in range(0, len(val_feats_dev), bs)]
                              ).cpu().numpy()
        labels = val_labels_dev.cpu().numpy()

        acc      = 100.0 * (preds == labels).mean()
        macro_f1 = 100.0 * f1_score(labels, preds, average="macro")
        key = f"{int(round(frac * 100))}pct"
        results[key] = {"top1_acc": round(acc, 2),
                        "macro_f1": round(macro_f1, 2)}
        print(f"  [{int(frac*100)}% labels]  "
              f"Top-1={acc:.2f}%  Macro-F1={macro_f1:.2f}%")

    return results


# =============================================================================
# §4.2  SEGMENTATION
# =============================================================================

class SegDecoder(nn.Module):
    def __init__(self, embed_dim, num_classes, img_size=160):
        super().__init__()
        self.img_size = img_size
        self.head     = nn.Conv2d(embed_dim, num_classes, kernel_size=1)

    def forward(self, global_feat):
        x = global_feat.unsqueeze(-1).unsqueeze(-1)
        x = self.head(x)
        return F.interpolate(x, size=(self.img_size, self.img_size),
                             mode="bilinear", align_corners=False)


def boundary_f1(pred_mask, gt_mask, num_classes, dilation=1):
    from scipy.ndimage import binary_dilation as bd
    bf1_list = []
    for c in range(num_classes):
        p = (pred_mask == c).astype(np.uint8)
        g = (gt_mask   == c).astype(np.uint8)
        if g.sum() == 0: continue
        p_b   = np.logical_xor(p, bd(p, iterations=dilation)).astype(np.uint8)
        g_b   = np.logical_xor(g, bd(g, iterations=dilation)).astype(np.uint8)
        inter = (p_b & g_b).sum(); denom = p_b.sum() + g_b.sum()
        if denom == 0: continue
        bf1_list.append(2 * inter / (denom + 1e-8))
    return float(np.mean(bf1_list)) if bf1_list else 0.0


def eval_segmentation(backbone_path, eurosat_dir, num_classes=10, epochs=30):
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone = load_backbone(backbone_path, device)
    split_dir = ensure_split(eurosat_dir, CFG["val_split"])
    _, val_tf = get_eval_transforms()
    aug_tf    = transforms.Compose([
        transforms.RandomResizedCrop(CFG["img_size"]),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    train_ds  = datasets.ImageFolder(
        os.path.join(split_dir, "train"), transform=aug_tf)
    val_ds    = datasets.ImageFolder(
        os.path.join(split_dir, "val"),   transform=val_tf)
    train_ldr = make_loader(train_ds, 64, shuffle=True,  drop_last=True)
    val_ldr   = make_loader(val_ds,   64, shuffle=False)

    decoder = SegDecoder(CFG["embed_dim"], num_classes,
                          img_size=CFG["img_size"]).to(device)
    opt     = torch.optim.Adam(decoder.parameters(), lr=1e-3)
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    use_amp = AMP_DTYPE is not None

    def get_global_feat(x):
        with torch.inference_mode():
            return backbone(x.to(device, non_blocking=True))

    for epoch in range(epochs):
        decoder.train()
        for x, y in train_ldr:
            feat = get_global_feat(x)
            y    = y.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=use_amp):
                logits = decoder(feat)
                target = y.view(-1, 1, 1).expand(
                    -1, CFG["img_size"], CFG["img_size"])
                loss   = F.cross_entropy(logits, target)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        if (epoch + 1) % 10 == 0:
            print(f"  Seg epoch {epoch+1}/{epochs}", flush=True)

    decoder.eval()
    confusion = np.zeros((num_classes, num_classes), dtype=np.int64)
    all_bf1   = []
    with torch.inference_mode():
        for x, y in val_ldr:
            feat  = get_global_feat(x)
            pred  = decoder(feat).argmax(1).cpu().numpy()
            label = y.numpy()
            for b in range(pred.shape[0]):
                gt_map = np.full_like(pred[b], label[b])
                for i in range(num_classes):
                    for j in range(num_classes):
                        confusion[i, j] += (
                            (gt_map == i) & (pred[b] == j)).sum()
                all_bf1.append(boundary_f1(pred[b], gt_map, num_classes))

    iou_per_class = []
    for c in range(num_classes):
        tp    = confusion[c, c]
        fp    = confusion[:, c].sum() - tp
        fn    = confusion[c, :].sum() - tp
        denom = tp + fp + fn
        if denom > 0:
            iou_per_class.append(tp / denom)

    miou     = float(np.mean(iou_per_class)) * 100
    mean_bf1 = float(np.mean(all_bf1)) * 100
    print(f"  Segmentation  mIoU={miou:.2f}%  Boundary-F1={mean_bf1:.2f}%")
    return {"miou": round(miou, 2), "boundary_f1": round(mean_bf1, 2)}


# =============================================================================
# §4.3  RETRIEVAL
# =============================================================================

def eval_retrieval(backbone_path, eurosat_dir, gps_csv=None,
                    ks=(1, 5, 10), geo_thresholds_km=(1, 5, 10)):
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone = load_backbone(backbone_path, device)
    split_dir = ensure_split(eurosat_dir, CFG["val_split"])
    _, val_tf = get_eval_transforms()

    val_ds  = datasets.ImageFolder(
        os.path.join(split_dir, "val"), transform=val_tf)
    val_ldr = make_loader(val_ds, 256, shuffle=False)

    print("  Extracting gallery features…")
    feats, labels = extract_features(backbone, val_ldr, device)
    feats = F.normalize(feats.float(), dim=-1)
    sim   = feats @ feats.T
    N     = feats.shape[0]

    results     = {}
    sim_no_diag = sim.clone()
    sim_no_diag.fill_diagonal_(-1e9)
    topk_max    = max(ks)
    top_indices = sim_no_diag.topk(topk_max, dim=1).indices

    for k in ks:
        top_k_labels = labels[top_indices[:, :k]]
        correct = (top_k_labels == labels.unsqueeze(1)).any(dim=1).sum().item()
        recall  = 100.0 * correct / N
        results[f"recall@{k}"] = round(recall, 2)
        print(f"  Recall@{k} = {recall:.2f}%")

    mAP = mean_average_precision(sim, labels)
    results["mAP"] = round(mAP, 2)
    print(f"  mAP = {mAP:.2f}%")

    if gps_csv is not None:
        import pandas as pd
        gps_df       = pd.read_csv(gps_csv)
        img_paths    = [val_ds.samples[i][0] for i in range(N)]
        fname_to_gps = {row["filename"]: (row["lat"], row["lon"])
                        for _, row in gps_df.iterrows()}
        errors_km = []
        geo_hits  = {k: {eps: 0 for eps in geo_thresholds_km} for k in ks}
        n_valid   = 0

        for i in range(N):
            q_fname = os.path.basename(img_paths[i])
            if q_fname not in fname_to_gps: continue
            n_valid += 1
            q_lat, q_lon = fname_to_gps[q_fname]
            top_fnames   = [os.path.basename(img_paths[j])
                            for j in top_indices[i, :topk_max].tolist()]
            if top_fnames[0] in fname_to_gps:
                r_lat, r_lon = fname_to_gps[top_fnames[0]]
                errors_km.append(haversine_km(q_lat, q_lon, r_lat, r_lon))
            for k in ks:
                cands = [f for f in top_fnames[:k] if f in fname_to_gps]
                for eps in geo_thresholds_km:
                    if any(haversine_km(q_lat, q_lon, *fname_to_gps[f]) <= eps
                           for f in cands):
                        geo_hits[k][eps] += 1

        if errors_km:
            med_err = float(np.median(errors_km)) * 1000
            p90_err = float(np.percentile(errors_km, 90)) * 1000
            results["median_loc_error_m"] = round(med_err, 1)
            results["p90_loc_error_m"]    = round(p90_err, 1)
            print(f"  Median loc. error = {med_err:.1f} m  "
                  f"(P90 = {p90_err:.1f} m)")

        if n_valid > 0:
            for k in ks:
                for eps in geo_thresholds_km:
                    r = 100.0 * geo_hits[k][eps] / n_valid
                    results[f"geo_recall@{k}_{eps}km"] = round(r, 2)
                    print(f"  Geo-Recall@{k} ({eps} km) = {r:.2f}%")

    return results


# =============================================================================
# §4.4  REPRESENTATION QUALITY
# =============================================================================

def eval_representation_quality(backbone_path, eurosat_dir,
                                  knn_k=20, num_classes=10):
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone = load_backbone(backbone_path, device)
    split_dir = ensure_split(eurosat_dir, CFG["val_split"])
    _, val_tf = get_eval_transforms()

    train_ds  = datasets.ImageFolder(
        os.path.join(split_dir, "train"), transform=val_tf)
    val_ds    = datasets.ImageFolder(
        os.path.join(split_dir, "val"),   transform=val_tf)
    train_ldr = make_loader(train_ds, 256, shuffle=False)
    val_ldr   = make_loader(val_ds,   256, shuffle=False)

    print("  Extracting features for representation quality…")
    train_feats, train_labels = extract_features(backbone, train_ldr, device)
    val_feats,   val_labels   = extract_features(backbone, val_ldr,   device)
    train_n = F.normalize(train_feats.float(), dim=-1)
    val_n   = F.normalize(val_feats.float(),   dim=-1)

    sim       = val_n @ train_n.T
    knn_preds = batched_knn_predict(
        sim.to(device), train_labels, knn_k, num_classes)
    knn_acc   = 100.0 * (knn_preds.cpu() == val_labels).float().mean().item()
    print(f"  kNN accuracy (k={knn_k}) = {knn_acc:.2f}%")

    eff_rank = effective_rank(val_feats.float())
    print(f"  Effective rank = {eff_rank:.1f}")

    unif = uniformity_score(val_n)
    print(f"  Uniformity = {unif:.4f}")

    aug = _base_aug(CFG["img_size"], CFG["global_scale"],
                    CFG["jitter_strength"], CFG["blur_prob"])
    two_view_ds  = TwoViewDataset(os.path.join(split_dir, "val"), aug)
    two_view_ldr = make_loader(two_view_ds, 256, shuffle=False)
    align_scores = []
    with torch.inference_mode():
        for x1, x2, _ in two_view_ldr:
            z1 = F.normalize(backbone(x1.to(device, non_blocking=True)), dim=-1)
            z2 = F.normalize(backbone(x2.to(device, non_blocking=True)), dim=-1)
            align_scores.append(
                (z1 - z2).pow(2).sum(dim=-1).mean().item())
    alignment = round(float(np.mean(align_scores)), 4)
    print(f"  Alignment = {alignment:.4f}")

    return {"knn_acc":        round(knn_acc, 2),
            "effective_rank": round(eff_rank, 1),
            "uniformity":     unif,
            "alignment":      alignment}


# =============================================================================
# §4.5  BAND MISMATCH ROBUSTNESS
# =============================================================================

class BandAdapterBackbone(nn.Module):
    def __init__(self, backbone, in_channels=13, out_channels=3):
        super().__init__()
        self.adapter  = nn.Conv2d(in_channels, out_channels,
                                   kernel_size=1, bias=False)
        self.backbone = backbone
        nn.init.kaiming_normal_(self.adapter.weight)

    def forward(self, x):
        return self.backbone(self.adapter(x))


def eval_band_mismatch(backbone_path, eurosat_rgb_dir,
                        eurosat_ms_dir=None, num_classes=10):
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone = load_backbone(backbone_path, device)
    split_dir = ensure_split(eurosat_rgb_dir, CFG["val_split"])
    _, val_tf = get_eval_transforms()

    val_ds  = datasets.ImageFolder(
        os.path.join(split_dir, "val"), transform=val_tf)
    val_ldr = make_loader(val_ds, 256, shuffle=False)
    feats, labels = extract_features(backbone, val_ldr, device)
    feats  = F.normalize(feats.float(), dim=-1)
    sim_nd = feats @ feats.T
    sim_nd.fill_diagonal_(-1e9)
    recall_rgb = 100.0 * (
        labels[sim_nd.argmax(dim=1)] == labels).float().mean().item()
    print(f"  Recall@1 (RGB, 3-band) = {recall_rgb:.2f}%")

    recall_ms = None
    if eurosat_ms_dir and os.path.exists(eurosat_ms_dir):
        ms_ds = MultiSpectralDataset(eurosat_ms_dir, n_channels=13)
        if len(ms_ds) == 0:
            print("  EuroSAT-MS: no files found — skipping.")
        else:
            adapter_bb   = BandAdapterBackbone(
                backbone, in_channels=13, out_channels=3).to(device)
            ms_ldr_train = make_loader(ms_ds, 128, shuffle=True)

            tmp_head  = nn.Linear(CFG["embed_dim"], num_classes).to(device)
            opt_adapt = torch.optim.AdamW(
                list(adapter_bb.adapter.parameters()) +
                list(tmp_head.parameters()), lr=1e-3)

            for _ in range(5):
                adapter_bb.adapter.train(); tmp_head.train()
                for x, y in ms_ldr_train:
                    x, y = (x.to(device, non_blocking=True),
                            y.to(device, non_blocking=True))
                    with torch.inference_mode():
                        feat = backbone(adapter_bb.adapter(x))
                    opt_adapt.zero_grad()
                    F.cross_entropy(tmp_head(feat), y).backward()
                    opt_adapt.step()
            del tmp_head

            adapter_bb.eval()
            ms_ldr_eval      = make_loader(ms_ds, 128, shuffle=False)
            ms_feats, ms_lbl = [], []
            with torch.inference_mode():
                for x, y in ms_ldr_eval:
                    ms_feats.append(
                        adapter_bb(x.to(device, non_blocking=True))
                        .float().cpu())
                    ms_lbl.append(y)
            ms_feats  = F.normalize(torch.cat(ms_feats), dim=-1)
            ms_labels = torch.cat(ms_lbl)
            sim_ms    = ms_feats @ ms_feats.T
            sim_ms.fill_diagonal_(-1e9)
            recall_ms = 100.0 * (
                ms_labels[sim_ms.argmax(dim=1)] == ms_labels
            ).float().mean().item()
            print(f"  Recall@1 (MS, 13-band) = {recall_ms:.2f}%")
    else:
        print("  EuroSAT-MS directory not found — skipping.")

    delta = round(recall_rgb - recall_ms, 2) if recall_ms is not None else None
    if delta is not None:
        print(f"  Band mismatch penalty Δ = {delta:.2f}%")

    return {
        "recall1_rgb":         round(recall_rgb, 2),
        "recall1_ms":          round(recall_ms, 2) if recall_ms is not None
                               else None,
        "band_mismatch_delta": delta,
    }


# =============================================================================
# ── Full evaluation orchestrator ──────────────────────────────────────────────
# =============================================================================

def run_full_evaluation(backbone_path, eurosat_rgb_dir=None,
                         eurosat_ms_dir=None, gps_csv=None):
    eurosat_rgb_dir = eurosat_rgb_dir or CFG["eurosat_rgb_dir"]
    eurosat_ms_dir  = eurosat_ms_dir  or CFG.get("eurosat_ms_dir")
    all_results     = {"model": "VICReg-ResNet50", "checkpoint": backbone_path}

    print("\n" + "=" * 60)
    print("§4.1  CLASSIFICATION (linear probe)")
    print("=" * 60)
    all_results["classification"] = eval_classification(
        backbone_path, eurosat_rgb_dir,
        label_fracs=(0.01, 0.1, 1.0), num_classes=CFG["num_classes"])

    print("\n" + "=" * 60)
    print("§4.2  SEGMENTATION")
    print("=" * 60)
    all_results["segmentation"] = eval_segmentation(
        backbone_path, eurosat_rgb_dir, num_classes=CFG["num_classes"])

    print("\n" + "=" * 60)
    print("§4.3  RETRIEVAL PERFORMANCE")
    print("=" * 60)
    all_results["retrieval"] = eval_retrieval(
        backbone_path, eurosat_rgb_dir, gps_csv=gps_csv,
        ks=CFG["retrieval_ks"], geo_thresholds_km=CFG["geo_thresholds_km"])

    print("\n" + "=" * 60)
    print("§4.4  REPRESENTATION QUALITY")
    print("=" * 60)
    all_results["representation"] = eval_representation_quality(
        backbone_path, eurosat_rgb_dir,
        knn_k=CFG["knn_k"], num_classes=CFG["num_classes"])

    print("\n" + "=" * 60)
    print("§4.5  BAND MISMATCH ROBUSTNESS")
    print("=" * 60)
    all_results["band_mismatch"] = eval_band_mismatch(
        backbone_path, eurosat_rgb_dir, eurosat_ms_dir)

    out_path = os.path.join(CFG["output_dir"], "vicreg_eval_results.json")
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nAll results saved to {out_path}")
    return all_results


# =============================================================================
# ── Entry point ───────────────────────────────────────────────────────────────
# =============================================================================

def is_notebook():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False


if __name__ == "__main__":
    world_size = torch.cuda.device_count()

    if world_size > 1 and not is_notebook():
        os.environ["MASTER_ADDR"] = "localhost"
        os.environ["MASTER_PORT"] = "12355"
        mp.spawn(train, args=(world_size,), nprocs=world_size, join=True)
    else:
        if world_size > 1 and is_notebook():
            print(f"Notebook detected — DDP disabled. "
                  f"Running single-GPU on cuda:0. "
                  f"({world_size} GPUs available but only 1 will be used.)")
        train(rank=0, world_size=1)

    BEST_CKPT = os.path.join(CFG["output_dir"],
                              f"vicreg_ep{CFG['epochs']}.pt")
    GPS_CSV   = f"{DATA_DIR}/eurosat/eurosat_gps.csv"
    run_full_evaluation(
        BEST_CKPT,
        CFG["eurosat_rgb_dir"],
        CFG["eurosat_ms_dir"],
        GPS_CSV if os.path.exists(GPS_CSV) else None,
    )

Notebook detected — DDP disabled. Running single-GPU on cuda:0. (2 GPUs available but only 1 will be used.)
Device: cuda:0  |  World: 1  |  Workers: 4  |  compile: False  |  AMP: torch.float16  |  albu: True
Stability config: lr=0.05  nu_cov=0.04  grad_clip=1.0  cov_clamp=50.0
Epoch [  1/200]  loss=23.2369  inv=0.5701  var=0.2793  cov=50.0000  lr=5.00e-03  time=134s
Epoch [  2/200]  loss=11.6519  inv=0.3365  var=0.0496  cov=50.0000  lr=1.00e-02  time=124s
Epoch [  3/200]  loss=9.5758  inv=0.2780  var=0.0250  cov=50.0000  lr=1.50e-02  time=124s
Epoch [  4/200]  loss=8.2467  inv=0.2329  var=0.0169  cov=50.0000  lr=2.00e-02  time=124s
Epoch [  5/200]  loss=7.3724  inv=0.1998  var=0.0151  cov=50.0000  lr=2.50e-02  time=124s
Epoch [  6/200]  loss=7.1004  inv=0.1878  var=0.0162  cov=50.0000  lr=3.00e-02  time=124s
Epoch [  7/200]  loss=6.8523  inv=0.1810  var=0.0131  cov=50.0000  lr=3.50e-02  time=123s
Epoch [  8/200]  loss=6.1187  inv=0.1548  var=0.0099  cov=50.0000  lr=4.00e-02  time=123s


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/vicreg/vicreg_ep200.pt'